# 30 — CI/CD & GitLab Pipelines

**Role:** Senior AWS DevOps Engineer | **Focus:** GitLab CI/CD, Multi-Layer Pipelines, Deployment Strategies

---

## Section A: GitLab CI Fundamentals

### Q1: Pipeline Architecture
**Question:** Design a GitLab CI pipeline for a monorepo containing a Python backend, React frontend, and Terraform infrastructure. How do you handle selective builds (only build what changed)?

**Expected Answer:**
- **`rules:changes`** to trigger jobs only when specific paths change.
- **Stages**: `lint` → `test` → `build` → `infra` → `deploy-staging` → `deploy-prod`.
- **Parent-child pipelines**: `trigger:include` to fan out to component-specific `.gitlab-ci.yml`.
- **Caching**: `cache:key` per branch for `node_modules`, `pip`, `.terraform`.
- **Artifacts**: Pass build outputs between stages (e.g., Docker image tag, Terraform plan file).

```yaml
# Example: selective trigger
backend-test:
  stage: test
  script: pytest
  rules:
    - changes:
        - backend/**/*
```

---

### Q2: GitLab Runners — Shared vs. Self-Hosted
**Question:** When would you use self-hosted GitLab Runners on AWS instead of shared runners? How do you scale them?

**Expected Answer:**
- **Self-hosted when**: Need VPC access (deploy to private subnets), compliance requirements, GPU builds, cost optimization at scale.
- **Scaling**: GitLab Runner with Docker Machine or Kubernetes executor + auto-scaling config.
- **EKS executor**: Run CI jobs as K8s pods — scales with the cluster, isolated per job.
- **Caching**: Use S3 as distributed cache backend for runners.
- **Security**: Runners in a dedicated CI/CD AWS account, assume roles to deploy cross-account.

---

### Q3: Secrets in CI/CD
**Question:** How do you securely inject AWS credentials and database passwords into a GitLab pipeline?

**Expected Answer:**
- **GitLab CI/CD Variables**: Masked + Protected variables for secrets.
- **OIDC Federation** (preferred): GitLab OIDC → AWS IAM Identity Provider → AssumeRoleWithWebIdentity. No long-lived credentials.
- **Vault integration**: HashiCorp Vault with GitLab JWT auth.
- **AWS Secrets Manager**: Fetch at runtime with `aws secretsmanager get-secret-value`.
- **Never**: Hardcode secrets in `.gitlab-ci.yml` or Dockerfiles.

## Section B: Deployment Strategies

### Q4: Blue/Green vs. Canary vs. Rolling
**Question:** Explain the difference between Blue/Green, Canary, and Rolling deployments. When would you use each for an EKS-hosted service?

**Expected Answer:**
- **Rolling**: Default K8s strategy. Gradually replaces old pods. Risk: both versions serve traffic during rollout.
- **Blue/Green**: Two full environments; switch traffic at the LB/DNS level. Zero-downtime, instant rollback. Cost: 2x resources during deploy.
- **Canary**: Route a small % of traffic (e.g., 5%) to new version. Monitor error rate. Gradually increase. Requires traffic splitting (Istio, ALB weighted target groups, Argo Rollouts).
- **EKS choice**: Argo Rollouts for Canary/Blue-Green with automatic analysis and rollback.

---

### Q5: Database Migrations in CI/CD
**Question:** How do you run database schema migrations (PostgreSQL/RDS) safely within a CI/CD pipeline?

**Expected Answer:**
- **Migration tool**: Alembic (Python), Flyway, or Liquibase.
- **Pipeline stage**: Run migrations BEFORE deploying new app code (forward-compatible schema changes).
- **Backward compatibility**: New schema must work with old code (expand-and-contract pattern).
- **Locking**: Avoid `ALTER TABLE` that locks tables for long periods. Use `CREATE INDEX CONCURRENTLY`.
- **Rollback plan**: Always write `down` migrations. Test rollback in staging.
- **Access**: Pipeline connects via VPC-internal endpoint; use IAM auth or Secrets Manager for credentials.

---

### Q6: Infrastructure Pipeline
**Question:** How do you safely run `terraform apply` in CI/CD without human oversight for every change?

**Expected Answer:**
- **MR workflow**: `terraform plan` runs automatically on MR; output posted as comment. `apply` requires manual approval (GitLab `when: manual` + protected environments).
- **Plan artifact**: Save the plan file as a GitLab artifact; apply uses the SAME plan (not re-planning).
- **Auto-apply for dev**: Non-destructive changes to dev can auto-apply. Prod always requires manual gate.
- **Policy checks**: Run `checkov` and `infracost` in the pipeline. Block MR if security issues found.
- **Blast radius**: Separate pipelines per component (networking vs. application infra).

## Section C: Advanced CI/CD Patterns

### Q7: Multi-Layer Pipeline Orchestration
**Question:** You need to deploy in this order: (1) Terraform infra, (2) Database migrations, (3) Backend service, (4) Frontend. How do you model dependencies in GitLab?

**Expected Answer:**
```yaml
stages:
  - infra
  - migrate
  - deploy-backend
  - deploy-frontend
  - smoke-test

terraform-apply:
  stage: infra
  when: manual  # gate for prod

db-migrate:
  stage: migrate
  needs: [terraform-apply]  # DAG dependency

deploy-api:
  stage: deploy-backend
  needs: [db-migrate]

deploy-web:
  stage: deploy-frontend
  needs: [deploy-api]

smoke:
  stage: smoke-test
  needs: [deploy-web]
  script: curl -f https://app.example.com/healthz
```
- **`needs` keyword**: Creates a DAG (Directed Acyclic Graph) for faster execution.
- **Fail-fast**: If `db-migrate` fails, downstream jobs are skipped.

---

### Q8: Docker Image Build & Registry
**Question:** How do you build and push Docker images in GitLab CI efficiently? How do you handle layer caching?

**Expected Answer:**
- **Multi-stage Dockerfiles**: Separate build and runtime stages to reduce image size.
- **Layer caching**: `docker build --cache-from` using the previously pushed image.
- **ECR**: Push to AWS ECR; authenticate with `aws ecr get-login-password`.
- **Tagging**: Tag with `$CI_COMMIT_SHA` (immutable) + `latest` for convenience. Never use `latest` in production K8s manifests.
- **Scanning**: `trivy` or ECR native scanning for CVEs before deployment.

---

### Q9: Pipeline Reliability & Flaky Tests
**Question:** Your CI pipeline fails intermittently due to flaky integration tests. How do you handle this without just adding `retry: 2`?

**Expected Answer:**
- **Quarantine**: Tag flaky tests, run them separately, don't block the main pipeline.
- **Root cause**: Usually shared state, timing issues, or external dependencies.
- **Fixes**: Use testcontainers for isolated DB/Redis per test, mock external APIs, use `wait-for-it` scripts.
- **Metrics**: Track flaky test rate over time; make it a team KPI.
- **`retry` is OK for infra flakiness** (network timeouts) but NOT for test logic.